In [137]:
import pandas as pd
import numpy as np

In [138]:
weights_driver = {
    'fare_amount': 10.0,
    'pickup_datetime': 8,
    'dropoff_datetime': 8,
    'payment_type': 1.0,
    'trip_distance_miles': 5,
    'pickup_latitude': 9.0,
    'pickup_longitude': 9.0,
    'dropoff_latitude': 9.0,
    'dropoff_longitude': 9.0,
    'passenger_count': 1.0
}
weights_passenger = {
    'fare_amount': 10.0,
    'trip_distance_miles': 10.0,
    'pickup_latitude': 7,
    'pickup_longitude': 7,
    'dropoff_latitude': 7,
    'dropoff_longitude': 7,
    'pickup_datetime': 8,
    'dropoff_datetime': 9,
    'payment_type': 1.0,
    'passenger_count': 1.0
}

In [139]:
def completeness_score(col, scenario):
    if scenario == 'driver':
        weight = weights_driver.get(col.name, 0)
    elif scenario == 'passenger':
        weight = weights_passenger.get(col.name, 0)
    else:
        weight = 0


    missing_count = col.isnull().sum()
    total_count = len(col)
    valid_count = total_count - missing_count
    completeness = (valid_count / total_count) ** weight if total_count > 0 else 1


    return max(completeness, 0)

def overall_completeness(data, scenario):
    scores = []
    for col in data.columns:
        score = completeness_score(data[col], scenario)
        scores.append(score)
    overall_score = np.mean(scores)
    return overall_score

In [140]:
import geopandas as gpd
from shapely.geometry import Point

# 1) Charger la limite de NYC (GeoJSON depuis ArcGIS)
nyc_url = ("https://services5.arcgis.com/GfwWNkhOj9bNBqoJ"
           "/arcgis/rest/services/NYC_Borough_Boundary_Water_Included"
           "/FeatureServer/0/query?where=1=1&outFields=*&f=geojson")
nyc = gpd.read_file(nyc_url)

# 2) Charger les polygones d'eau (shapefile téléchargé)
water = gpd.read_file("tl_2023_36061_areawater.shp")

# 3) Mettre les deux jeux de données dans la même projection (lat/lon EPSG:4326)
nyc = nyc.to_crs(epsg=4326)
water = water.to_crs(epsg=4326)

NYC_BBOX = {
    'lat_min': 40.4,
    'lat_max': 41.0,
    'lon_min': -74.4,
    'lon_max': -73.6
}

def check_location(lat, lon):
    pt = Point(lon, lat)

    # Vérifier l'eau en premier
    if water.contains(pt).any():
        return False

    # Si pas dans l'eau, vérifier NYC
    if nyc.contains(pt).any():
        return True
    
    if (NYC_BBOX['lat_min'] <= lat <= NYC_BBOX['lat_max'] and
        NYC_BBOX['lon_min'] <= lon <= NYC_BBOX['lon_max']):
        return True

    # Si ni eau ni NYC
    return False

In [141]:
def accuracy_score(col, data, scenario):        
    if scenario == 'driver':
        weight = weights_driver.get(col.name, 0)
    elif scenario == 'passenger':
        weight = weights_passenger.get(col.name, 0)
    else:
        weight = 0
    
    # Get non-null values (universe for accuracy calculation)
    not_null = col.notna()
    non_null_count = not_null.sum()
    
    # If no non-null values, return NaN (not applicable)
    if non_null_count == 0:
        return np.nan
    
    valid_count = 0
    
    # Define validation rules per column
    if type(col) != pd.DataFrame and col.name == 'fare_amount':
        # Range: >= 2.50 and <= 500
        if(col.dtype == 'object'):
            col = pd.to_numeric(col, errors='coerce')
        valid = (col >= 2.50) & (col <= 500)
        valid_count = (valid & not_null).sum()
    
    elif type(col) != pd.DataFrame and col.name == 'trip_distance_miles':
        # Range: > 0 and <= 100
        if(col.dtype == 'object'):
            col = pd.to_numeric(col, errors='coerce')
        valid = (col > 0) & (col <= 100)
        valid_count = (valid & not_null).sum()
    
    elif type(col) != pd.DataFrame and col.name == 'passenger_count':
        # Range: >= 1 and <= 5
        if(col.dtype == 'object'):
            col = pd.to_numeric(col, errors='coerce')
        valid = (col >= 1) & (col <= 5)
        valid_count = (valid & not_null).sum()
    
    elif type(col) != pd.DataFrame and col.name in ['pickup_datetime', 'dropoff_datetime']:
        # Check if date can be parsed and is within valid range
        parsed = pd.to_datetime(col, errors='coerce')
        valid = (parsed >= pd.Timestamp('2020-01-01')) & (parsed <= pd.Timestamp('2024-12-31'))
        valid_count = (valid & not_null).sum()
    
    elif col.name in ['pickup_latitude', 'pickup_longitude']:
        # Consistency: pickup coordinates should be within NYC and not in water
        if 'pickup_latitude' in data.columns and 'pickup_longitude' in data.columns:
            points = []
            for i in range(len(data)):
                plat = float(data[['pickup_latitude', 'pickup_longitude']].iloc[i]["pickup_latitude"])
                plon = float(data[['pickup_latitude', 'pickup_longitude']].iloc[i]["pickup_longitude"])
                points.append((plat, plon))
            valid = [check_location(pt[0], pt[1]) for pt in points]
            valid = pd.Series(valid)
            not_null = data[['pickup_latitude', 'pickup_longitude']].notna().all(axis=1)
            valid_count = (valid & not_null).sum()
        else:
            return np.nan
    elif col.name in ['dropoff_latitude', 'dropoff_longitude']:
        # Consistency: dropoff coordinates should be within NYC and not in water
        if 'dropoff_latitude' in data.columns and 'dropoff_longitude' in data.columns:
            points = []
            for i in range(len(data)):
                dlat = float(data[['dropoff_latitude', 'dropoff_longitude']].iloc[i]["dropoff_latitude"])
                dlon = float(data[['dropoff_latitude', 'dropoff_longitude']].iloc[i]["dropoff_longitude"])
                points.append((dlat, dlon))
            valid = [check_location(pt[0], pt[1]) for pt in points]
            valid = pd.Series(valid)
            not_null = data[['dropoff_latitude', 'dropoff_longitude']].notna().all(axis=1)
            valid_count = (valid & not_null).sum()
        else:
            return np.nan
    else:
        # For unknown columns, consider all non-null values as accurate
        valid_count = non_null_count
    
    # Calculate accuracy as percentage of valid values among non-null values
    accuracy = (valid_count / non_null_count) ** weight if non_null_count > 0 else 1    
    
    return round(accuracy, 2)

def overall_accuracy(df, scenario):
    scores = []
    for col in df.columns:
        score = accuracy_score(df[col], df, scenario)
        if not np.isnan(score):  # Ignore NaN (not applicable) columns
            scores.append(score)
    
    overall_score = np.mean(scores) if scores else 0
    return round(overall_score, 2)

In [142]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calcule la distance en vol d'oiseau (haversine) en miles"""
    R = 3958.8  # Rayon de la Terre en miles
    
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return R * c

In [143]:
def consistency_score(col, df, scenario):
    """
    Calculate consistency score for a column based on logical relationships with other columns.
    Consistency checks inter-attribute relationships, not single column values.
    
    :param col: pandas Series (column from dataframe)
    :param df: full pandas DataFrame (needed for relationship checks)
    :param scenario: 'driver' or 'passenger'
    :return: float between 0 and 1 (consistency score)
    """
    
    if scenario == 'driver':
        weight = weights_driver.get(col.name, 0)
    elif scenario == 'passenger':
        weight = weights_passenger.get(col.name, 0)
    else:
        weight = 0
    
    valid_count = 0
    total_count = len(df)
    
    # Define consistency rules per column (relationships with other columns)
    
    if col.name == 'pickup_datetime':
        # Consistency: pickup_datetime < dropoff_datetime
        if 'dropoff_datetime' in df.columns:
            p = pd.to_datetime(df['pickup_datetime'], errors='coerce')
            d = pd.to_datetime(df['dropoff_datetime'], errors='coerce')
            subset = p.notna() & d.notna()
            
            is_consistent = d > p
            valid_count = (is_consistent & subset).sum()
            total_count = subset.sum()
        else:
            return np.nan
    
    elif col.name == 'dropoff_datetime':
        # Consistency: dropoff_datetime > pickup_datetime
        if 'pickup_datetime' in df.columns:
            p = pd.to_datetime(df['pickup_datetime'], errors='coerce')
            d = pd.to_datetime(df['dropoff_datetime'], errors='coerce')
            subset = p.notna() & d.notna()
            
            is_consistent = d > p
            valid_count = (is_consistent & subset).sum()
            total_count = subset.sum()
        else:
            return np.nan
    
    elif col.name == 'fare_amount':
        # Consistency: fare_amount should be coherent with trip_distance_miles
        # Rule: if distance > 0, fare/distance ratio should be reasonable (< 5 $/mile)
        if 'trip_distance_miles' in df.columns:
            subset = (pd.to_numeric(df['trip_distance_miles'], errors='coerce') > 0) & \
                     (df['trip_distance_miles'].notna()) & \
                     (df['fare_amount'].notna())
            
            if subset.sum() == 0:
                return np.nan
            
            ratio = pd.to_numeric(df.loc[subset, 'fare_amount'], errors='coerce') / pd.to_numeric(df.loc[subset, 'trip_distance_miles'], errors='coerce')
            is_consistent = ratio < 132  # Max 132$ per mile is reasonable
            
            valid_count = is_consistent.sum()
            total_count = subset.sum()
        else:
            return np.nan
    
    elif col.name == 'trip_distance_miles':
        # Consistency: trip distance should be coherent with fare_amount
        # Rule: if distance > 0, fare/distance ratio should be reasonable (< 5 $/mile)
        if 'fare_amount' in df.columns:
            subset = (pd.to_numeric(df['trip_distance_miles'], errors='coerce') > 0) & \
                     (df['trip_distance_miles'].notna()) & \
                     (df['fare_amount'].notna())
            
            if subset.sum() == 0:
                return np.nan
            
            ratio = pd.to_numeric(df.loc[subset, 'fare_amount'], errors='coerce') / pd.to_numeric(df.loc[subset, 'trip_distance_miles'], errors='coerce')
            is_consistent = ratio < 132  # Max 132$ per mile is reasonable
            
            valid_count = is_consistent.sum()
            total_count = subset.sum()
        else:
            return np.nan
    
    elif col.name in ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']:
        # Consistency: Distance en vol d'oiseau <= trip_distance_miles
        required_cols = ['pickup_latitude', 'pickup_longitude', 
                        'dropoff_latitude', 'dropoff_longitude', 
                        'trip_distance_miles']
        
        if all(c in df.columns for c in required_cols):
            subset = (df['pickup_latitude'].notna() & 
                     df['pickup_longitude'].notna() &
                     df['dropoff_latitude'].notna() & 
                     df['dropoff_longitude'].notna() &
                     df['trip_distance_miles'].notna() &
                     (pd.to_numeric(df['trip_distance_miles'], errors='coerce') > 0))
            
            if subset.sum() == 0:
                return np.nan
            
            # Calculer la distance haversine pour chaque ligne
            haversine_dist = haversine_distance(
                df.loc[subset, 'pickup_latitude'].values,
                df.loc[subset, 'pickup_longitude'].values,
                df.loc[subset, 'dropoff_latitude'].values,
                df.loc[subset, 'dropoff_longitude'].values
            )
            
            # Vérifier que distance haversine <= distance réelle
            epsilon = 0.01  # Tolérance pour les petites erreurs
            is_consistent = haversine_dist <= pd.to_numeric(df.loc[subset, 'trip_distance_miles'], errors='coerce').values + epsilon
            
            valid_count = is_consistent.sum()
            total_count = subset.sum()
        else:
            return np.nan
    
    else:
        # For columns without specific consistency rules, return NaN (not applicable)
        return np.nan
    
    # Avoid division by zero
    if total_count == 0:
        return np.nan
    
    # Calculate consistency as percentage of consistent values
    consistency = (valid_count / total_count) ** weight if total_count > 0 else 1
    return round(consistency, 2)

def overall_consistency(df, scenario):
    """
    Calculate overall consistency score across applicable columns for a given scenario.
    
    :param df: pandas DataFrame
    :param scenario: 'driver' or 'passenger'
    :return: float representing overall consistency score
    """
    scores = []
    for col in df.columns:
        score = consistency_score(df[col], df, scenario)
        if not np.isnan(score):  # Ignore NaN (not applicable) columns
            scores.append(score)
    
    # Average only columns with consistency rules defined
    overall_score = np.mean(scores) if scores else 0
    return round(overall_score, 2)

In [144]:
def uniqueness_score(col, data, scenario):
    """
    Calculate uniqueness score for a column based on the proportion of unique values.
    
    :param col: pandas Series (column from dataframe)
    :param data: pandas DataFrame (the full dataset)
    :param scenario: 'driver' or 'passenger'
    :return: float between 0 and 1 (uniqueness score)
    """
    if scenario == 'driver':
        weight = weights_driver.get(col.name, 0)
    elif scenario == 'passenger':
        weight = weights_passenger.get(col.name, 0)
    else:
        weight = 0

    total_count = len(col)
    if total_count == 0:
        return 1.0  # If no data, consider uniqueness as perfect
    uniqueness = (len(data.drop_duplicates()) / total_count) ** weight
    
    return round(uniqueness, 2)

def overall_uniqueness(df, scenario):
    """
    Calculate overall uniqueness score across applicable columns for a given scenario.
    
    :param df: pandas DataFrame
    :param scenario: 'driver' or 'passenger'
    :return: float representing overall uniqueness score
    """
    scores = []
    for col in df.columns:
        score = uniqueness_score(df[col], df, scenario)
        if not np.isnan(score):  # Ignore NaN (not applicable) columns
            scores.append(score)

    # Average only columns with uniqueness rules defined
    overall_score = np.mean(scores) if scores else 0
    return round(overall_score, 2)

In [145]:
def assessment(data):
    results = pd.DataFrame()
    columns = data.columns.tolist()
    scenarios = ['driver', 'passenger']

    for col in columns:
        for scenario in scenarios:
            results.at[col, scenario + '_accuracy'] = accuracy_score(data[col], data, scenario)

    for col in columns:
        for scenario in scenarios:
            results.at[col, scenario + '_completeness'] = completeness_score(data[col], scenario)

    for col in columns:
        for scenario in scenarios:
            results.at[col, scenario + '_consistency'] = consistency_score(data[col], data, scenario)

    for col in columns:
        for scenario in scenarios:
            results.at[col, scenario + '_uniqueness'] = uniqueness_score(data[col], data, scenario)

    return results

In [146]:
original = pd.read_csv("Taxi.csv")
modified = pd.read_csv("Taxi_modifié.csv")
corrigé = pd.read_csv("Taxi_corrigé.csv")

In [147]:
assessment_original = assessment(original)
assessment_modified = assessment(modified)
assessment_corrigé = assessment(corrigé)

In [148]:
assessment_original

,driver_accuracy,passenger_accuracy,driver_completeness,passenger_completeness,driver_consistency,passenger_consistency,driver_uniqueness,passenger_uniqueness
pickup_datetime,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0
dropoff_datetime,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0
pickup_latitude,0.8,0.84,1.0,1.0,1.0,1.0,1.0,1.0
pickup_longitude,0.8,0.84,1.0,1.0,1.0,1.0,1.0,1.0
dropoff_latitude,0.8,0.84,1.0,1.0,1.0,1.0,1.0,1.0
dropoff_longitude,0.8,0.84,1.0,1.0,1.0,1.0,1.0,1.0
trip_distance_miles,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0
fare_amount,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0
passenger_count,1.0,1.00,1.0,1.0,NaN,NaN,1.0,1.0
payment_type,1.0,1.00,1.0,1.0,NaN,NaN,1.0,1.0


In [149]:
assessment_modified

,driver_accuracy,passenger_accuracy,driver_completeness,passenger_completeness,driver_consistency,passenger_consistency,driver_uniqueness,passenger_uniqueness
pickup_datetime,0.88,0.88,1.000000,1.000000,0.88,0.88,0.64,0.64
dropoff_datetime,1.00,1.00,1.000000,1.000000,0.88,0.87,0.64,0.61
pickup_latitude,0.78,0.82,1.000000,1.000000,0.95,0.96,0.61,0.68
pickup_longitude,0.78,0.82,1.000000,1.000000,0.95,0.96,0.61,0.68
dropoff_latitude,0.76,0.81,1.000000,1.000000,0.95,0.96,0.61,0.68
dropoff_longitude,0.76,0.81,1.000000,1.000000,0.95,0.96,0.61,0.68
trip_distance_miles,0.96,0.92,0.887508,0.787671,1.00,1.00,0.76,0.57
fare_amount,0.92,0.92,0.759593,0.759593,1.00,1.00,0.57,0.57
passenger_count,1.00,1.00,1.000000,1.000000,NaN,NaN,0.95,0.95
payment_type,1.00,1.00,1.000000,1.000000,NaN,NaN,0.95,0.95


In [150]:
assessment_corrigé

,driver_accuracy,passenger_accuracy,driver_completeness,passenger_completeness,driver_consistency,passenger_consistency,driver_uniqueness,passenger_uniqueness
pickup_datetime,1.00,1.00,1.0,1.0,1.00,1.00,1.0,1.0
dropoff_datetime,1.00,1.00,1.0,1.0,1.00,1.00,1.0,1.0
pickup_latitude,0.89,0.92,1.0,1.0,0.97,0.97,1.0,1.0
pickup_longitude,0.89,0.92,1.0,1.0,0.97,0.97,1.0,1.0
dropoff_latitude,0.95,0.96,1.0,1.0,0.97,0.97,1.0,1.0
dropoff_longitude,0.95,0.96,1.0,1.0,0.97,0.97,1.0,1.0
trip_distance_miles,1.00,1.00,1.0,1.0,1.00,1.00,1.0,1.0
fare_amount,1.00,1.00,1.0,1.0,1.00,1.00,1.0,1.0
passenger_count,1.00,1.00,1.0,1.0,NaN,NaN,1.0,1.0
payment_type,1.00,1.00,1.0,1.0,NaN,NaN,1.0,1.0


In [151]:
scenarios = ['driver', 'passenger']
overrall_results = pd.DataFrame()
for scenario in scenarios:
    overrall_results.at["completeness", scenario + "_original"] = overall_completeness(original, scenario)
    overrall_results.at["accuracy", scenario + "_original"] = overall_accuracy(df=original, scenario=scenario)
    overrall_results.at["consistency", scenario + "_original"] = overall_consistency(original, scenario)
    overrall_results.at["uniqueness", scenario + "_original"] = overall_uniqueness(original, scenario)

    overrall_results.at["completeness", scenario + "_modified"] = overall_completeness(modified, scenario)
    overrall_results.at["accuracy", scenario + "_modified"] = overall_accuracy(df=modified, scenario=scenario)
    overrall_results.at["consistency", scenario + "_modified"] = overall_consistency(modified, scenario)
    overrall_results.at["uniqueness", scenario + "_modified"] = overall_uniqueness(modified, scenario)

    overrall_results.at["completeness", scenario + "_corrigé"] = overall_completeness(corrigé, scenario)
    overrall_results.at["accuracy", scenario + "_corrigé"] = overall_accuracy(df=corrigé, scenario=scenario)
    overrall_results.at["consistency", scenario + "_corrigé"] = overall_consistency(corrigé, scenario)
    overrall_results.at["uniqueness", scenario + "_corrigé"] = overall_uniqueness(corrigé, scenario)

overrall_results.at["completeness", "improvement"] = max(overrall_results.at["completeness", "driver_corrigé"] - overrall_results.at["completeness", "driver_modified"], overrall_results.at["completeness", "passenger_corrigé"] - overrall_results.at["completeness", "passenger_modified"])
overrall_results.at["accuracy", "improvement"] = max(overrall_results.at["accuracy", "driver_corrigé"] - overrall_results.at["accuracy", "driver_modified"], overrall_results.at["accuracy", "passenger_corrigé"] - overrall_results.at["accuracy", "passenger_modified"])
overrall_results.at["consistency", "improvement"] = max(overrall_results.at["consistency", "driver_corrigé"] - overrall_results.at["consistency", "driver_modified"], overrall_results.at["consistency", "passenger_corrigé"] - overrall_results.at["consistency", "passenger_modified"])
overrall_results.at["uniqueness", "improvement"] = max(overrall_results.at["uniqueness", "driver_corrigé"] - overrall_results.at["uniqueness", "driver_modified"], overrall_results.at["uniqueness", "passenger_corrigé"] - overrall_results.at["uniqueness", "passenger_modified"])

overrall_results

,driver_original,driver_modified,driver_corrigé,passenger_original,passenger_modified,passenger_corrigé,improvement
completeness,1.00,0.96471,1.00,1.00,0.954726,1.00,0.045274
accuracy,0.92,0.88000,0.97,0.94,0.900000,0.98,0.090000
consistency,1.00,0.94000,0.98,1.00,0.950000,0.98,0.040000
uniqueness,1.00,0.70000,1.00,1.00,0.700000,1.00,0.300000


In [152]:
corrupted = pd.read_csv("df_corrupted.csv")
assessment_corrupted = assessment(corrupted)

In [153]:
corrupted_corrigé = pd.read_csv("Taxi_corrupted_corrigé.csv")
assessment_corrupted_corrigé = assessment(corrupted_corrigé)

In [154]:
assessment_corrupted

,driver_accuracy,passenger_accuracy,driver_completeness,passenger_completeness,driver_consistency,passenger_consistency,driver_uniqueness,passenger_uniqueness
pickup_datetime,0.58,0.58,0.629297,0.629297,0.93,0.93,1.0,1.0
dropoff_datetime,0.58,0.54,0.609569,0.572995,0.93,0.92,1.0,1.0
pickup_latitude,0.49,0.57,0.593899,0.666805,1.00,1.00,1.0,1.0
pickup_longitude,0.46,0.55,0.630249,0.698337,1.00,1.00,1.0,1.0
dropoff_latitude,0.56,0.63,0.622825,0.691931,1.00,1.00,1.0,1.0
dropoff_longitude,0.54,0.62,0.637752,0.704795,1.00,1.00,1.0,1.0
trip_distance_miles,0.80,0.63,0.778885,0.606662,0.63,0.39,1.0,1.0
fare_amount,0.37,0.37,0.567960,0.567960,0.39,0.39,1.0,1.0
passenger_count,0.96,0.96,0.925000,0.925000,NaN,NaN,1.0,1.0
payment_type,1.00,1.00,0.948750,0.948750,NaN,NaN,1.0,1.0


In [155]:
assessment_corrupted_corrigé

,driver_accuracy,passenger_accuracy,driver_completeness,passenger_completeness,driver_consistency,passenger_consistency,driver_uniqueness,passenger_uniqueness
pickup_datetime,1.00,1.00,0.970391,0.970391,1.0,1.0,1.0,1.0
dropoff_datetime,1.00,1.00,0.970391,0.966752,1.0,1.0,1.0,1.0
pickup_latitude,0.82,0.86,0.796236,0.837592,1.0,1.0,1.0,1.0
pickup_longitude,0.72,0.78,0.903189,0.923859,1.0,1.0,1.0,1.0
dropoff_latitude,0.80,0.84,0.882842,0.907631,1.0,1.0,1.0,1.0
dropoff_longitude,0.85,0.88,0.833748,0.868126,1.0,1.0,1.0,1.0
trip_distance_miles,1.00,1.00,0.993766,0.987570,1.0,1.0,1.0,1.0
fare_amount,1.00,1.00,0.987570,0.987570,1.0,1.0,1.0,1.0
passenger_count,0.96,0.96,1.000000,1.000000,NaN,NaN,1.0,1.0
payment_type,1.00,1.00,1.000000,1.000000,NaN,NaN,1.0,1.0


In [156]:
overrall_corrupted = pd.DataFrame()
for scenario in scenarios:
    overrall_corrupted.at["completeness", scenario + "_corrupted"] = overall_completeness(corrupted, scenario)
    overrall_corrupted.at["accuracy", scenario + "_corrupted"] = overall_accuracy(df=corrupted, scenario=scenario)
    overrall_corrupted.at["consistency", scenario + "_corrupted"] = overall_consistency(corrupted, scenario)
    overrall_corrupted.at["uniqueness", scenario + "_corrupted"] = overall_uniqueness(corrupted, scenario)

In [157]:
overrall_corrupted_corrige = pd.DataFrame()
for scenario in scenarios:
    overrall_corrupted_corrige.at["completeness", scenario + "_corrupted"] = overall_completeness(corrupted_corrigé, scenario)
    overrall_corrupted_corrige.at["accuracy", scenario + "_corrupted"] = overall_accuracy(df=corrupted_corrigé, scenario=scenario)
    overrall_corrupted_corrige.at["consistency", scenario + "_corrupted"] = overall_consistency(corrupted_corrigé, scenario)
    overrall_corrupted_corrige.at["uniqueness", scenario + "_corrupted"] = overall_uniqueness(corrupted_corrigé, scenario)

In [158]:
overrall_corrupted

,driver_corrupted,passenger_corrupted
completeness,0.694419,0.701253
accuracy,0.630000,0.640000
consistency,0.860000,0.830000
uniqueness,1.000000,1.000000


In [159]:
overrall_corrupted_corrige

,driver_corrupted,passenger_corrupted
completeness,0.933813,0.944949
accuracy,0.910000,0.930000
consistency,1.000000,1.000000
uniqueness,1.000000,1.000000


In [160]:
overrall_results

,driver_original,driver_modified,driver_corrigé,passenger_original,passenger_modified,passenger_corrigé,improvement
completeness,1.00,0.96471,1.00,1.00,0.954726,1.00,0.045274
accuracy,0.92,0.88000,0.97,0.94,0.900000,0.98,0.090000
consistency,1.00,0.94000,0.98,1.00,0.950000,0.98,0.040000
uniqueness,1.00,0.70000,1.00,1.00,0.700000,1.00,0.300000


In [161]:
df_results = overrall_results.copy()
df_results["driver_corrupted"] = overrall_corrupted["driver_corrupted"]
df_results["passenger_corrupted"] = overrall_corrupted["passenger_corrupted"]
df_results["driver_corrupted_corrige"] = overrall_corrupted_corrige["driver_corrupted"]
df_results["passenger_corrupted_corrige"] = overrall_corrupted_corrige["passenger_corrupted"]
df_results

,driver_original,driver_modified,driver_corrigé,passenger_original,passenger_modified,passenger_corrigé,improvement,driver_corrupted,passenger_corrupted,driver_corrupted_corrige,passenger_corrupted_corrige
completeness,1.00,0.96471,1.00,1.00,0.954726,1.00,0.045274,0.694419,0.701253,0.933813,0.944949
accuracy,0.92,0.88000,0.97,0.94,0.900000,0.98,0.090000,0.630000,0.640000,0.910000,0.930000
consistency,1.00,0.94000,0.98,1.00,0.950000,0.98,0.040000,0.860000,0.830000,1.000000,1.000000
uniqueness,1.00,0.70000,1.00,1.00,0.700000,1.00,0.300000,1.000000,1.000000,1.000000,1.000000
